## Bộ dữ liệu ví dụ `weather`

Đây là bộ dữ liệu rất nổi tiếng khi học decision tree. Mỗi dòng mô tả tình huống thời tiết và cột `play` là quyết định cuối cùng:
- `yes`: nên chơi,
- `no`: không nên chơi.

Các thuộc tính đều là categorical, nên rất hợp với ID3.

In [1]:
weather_data = [
    {'outlook': 'sunny',    'temperature': 'hot',  'humidity': 'high',   'wind': 'weak',   'play': 'no'},
    {'outlook': 'sunny',    'temperature': 'hot',  'humidity': 'high',   'wind': 'strong', 'play': 'no'},
    {'outlook': 'overcast', 'temperature': 'hot',  'humidity': 'high',   'wind': 'weak',   'play': 'yes'},
    {'outlook': 'rainy',    'temperature': 'mild', 'humidity': 'high',   'wind': 'weak',   'play': 'yes'},
    {'outlook': 'rainy',    'temperature': 'cool', 'humidity': 'normal', 'wind': 'weak',   'play': 'yes'},
    {'outlook': 'rainy',    'temperature': 'cool', 'humidity': 'normal', 'wind': 'strong', 'play': 'no'},
    {'outlook': 'overcast', 'temperature': 'cool', 'humidity': 'normal', 'wind': 'strong', 'play': 'yes'},
    {'outlook': 'sunny',    'temperature': 'mild', 'humidity': 'high',   'wind': 'weak',   'play': 'no'},
    {'outlook': 'sunny',    'temperature': 'cool', 'humidity': 'normal', 'wind': 'weak',   'play': 'yes'},
    {'outlook': 'rainy',    'temperature': 'mild', 'humidity': 'normal', 'wind': 'weak',   'play': 'yes'},
    {'outlook': 'sunny',    'temperature': 'mild', 'humidity': 'normal', 'wind': 'strong', 'play': 'yes'},
    {'outlook': 'overcast', 'temperature': 'mild', 'humidity': 'high',   'wind': 'strong', 'play': 'yes'},
    {'outlook': 'overcast', 'temperature': 'hot',  'humidity': 'normal', 'wind': 'weak',   'play': 'yes'},
    {'outlook': 'rainy',    'temperature': 'mild', 'humidity': 'high',   'wind': 'strong', 'play': 'no'}
]

weather_data[:3]

[{'outlook': 'sunny',
  'temperature': 'hot',
  'humidity': 'high',
  'wind': 'weak',
  'play': 'no'},
 {'outlook': 'sunny',
  'temperature': 'hot',
  'humidity': 'high',
  'wind': 'strong',
  'play': 'no'},
 {'outlook': 'overcast',
  'temperature': 'hot',
  'humidity': 'high',
  'wind': 'weak',
  'play': 'yes'}]

## Entropy là gì?

Trong ID3, ta cần một cách đo xem một node đang "lẫn lộn" đến mức nào.

Nếu một node toàn là `yes` hoặc toàn là `no` thì node đó rất tinh khiết, entropy bằng `0`.

Nếu tỉ lệ giữa các class trộn lẫn nhiều hơn, entropy tăng lên. Vì vậy ID3 sẽ cố chọn thuộc tính giúp entropy giảm mạnh nhất sau khi chia.

In [2]:
from collections import Counter
import math

def entropy(labels):
    counts = Counter(labels)
    total = len(labels)
    ent = 0.0
    for count in counts.values():
        p = count / total
        ent -= p * math.log2(p)
    return ent

print('Entropy([yes, yes, yes]) =', round(entropy(['yes', 'yes', 'yes']), 4))
print('Entropy([yes, no])       =', round(entropy(['yes', 'no']), 4))
print('Entropy of play column  =', round(entropy([row['play'] for row in weather_data]), 4))

Entropy([yes, yes, yes]) = 0.0
Entropy([yes, no])       = 1.0
Entropy of play column  = 0.9403


## Information Gain

ID3 dùng **information gain** để chọn thuộc tính tốt nhất ở mỗi bước.

Ý tưởng rất trực giác:
- entropy trước khi chia càng cao càng lẫn,
- sau khi chia mà các nhóm con tinh khiết hơn thì entropy trung bình giảm,
- thuộc tính nào làm entropy giảm nhiều nhất thì được chọn trước.

Công thức là:
`Information Gain = Entropy(trước khi chia) - Entropy(trung bình sau khi chia)`

In [3]:
def partition_by_attribute(rows, attribute):
    groups = {}
    for row in rows:
        value = row[attribute]
        groups.setdefault(value, []).append(row)
    return groups

def dataset_entropy(rows, target='play'):
    return entropy([row[target] for row in rows])

def information_gain(rows, attribute, target='play'):
    base_entropy = dataset_entropy(rows, target=target)
    groups = partition_by_attribute(rows, attribute)
    weighted_entropy = 0.0
    total = len(rows)
    for subset in groups.values():
        weighted_entropy += (len(subset) / total) * dataset_entropy(subset, target=target)
    return base_entropy - weighted_entropy


## Xem thuộc tính nào tốt nhất ở node gốc

Cell này rất quan trọng vì nó cho mình thấy quyết định đầu tiên của ID3 không phải ngẫu nhiên.

Ta thử tính information gain cho từng thuộc tính:
- `outlook`
- `temperature`
- `humidity`
- `wind`

Thuộc tính có gain cao nhất sẽ được chọn làm node gốc.

In [4]:
attributes = ['outlook', 'temperature', 'humidity', 'wind']
for attribute in attributes:
    print(f'{attribute:12s}: {information_gain(weather_data, attribute):.4f}')

outlook     : 0.2467
temperature : 0.0292
humidity    : 0.1518
wind        : 0.0481


## Cài đặt node của cây quyết định

Để cây dễ thao tác, mình biểu diễn mỗi node bằng một lớp nhỏ.

Một node có thể là:
- **leaf node**: chứa nhãn dự đoán cuối cùng,
- **non-leaf node**: chứa thuộc tính cần hỏi tiếp và các nhánh con.

Mình thêm luôn hàm `pretty()` để in cây ra theo dạng văn bản cho dễ đọc.

In [5]:
class TreeNode:
    def __init__(self, attribute=None, label=None, default_label=None):
        self.attribute = attribute
        self.label = label
        self.default_label = default_label
        self.children = {}

    def is_leaf(self):
        return self.label is not None

    def predict(self, sample):
        if self.is_leaf():
            return self.label
        value = sample.get(self.attribute)
        child = self.children.get(value)
        if child is None:
            return self.default_label
        return child.predict(sample)

    def pretty(self, level=0):
        indent = '  ' * level
        if self.is_leaf():
            return indent + f'Predict -> {self.label}'

        lines = [indent + f'[{self.attribute}]']
        for value, child in self.children.items():
            lines.append(indent + f'  ({value})')
            lines.append(child.pretty(level + 2))
        return '\n'.join(lines)

def majority_label(rows, target='play'):
    counts = Counter(row[target] for row in rows)
    return counts.most_common(1)[0][0]

def all_same_label(rows, target='play'):
    labels = [row[target] for row in rows]
    return len(set(labels)) == 1


## Cài đặt thuật toán ID3

Bây giờ là phần cốt lõi.

ID3 làm việc theo kiểu đệ quy:
1. Nếu node hiện tại đã toàn cùng một nhãn thì dừng và tạo leaf.
2. Nếu đã hết thuộc tính để hỏi thì trả về nhãn chiếm đa số.
3. Tính information gain cho các thuộc tính còn lại.
4. Chọn thuộc tính tốt nhất.
5. Chia dữ liệu theo từng giá trị của thuộc tính đó rồi gọi đệ quy tiếp cho từng nhánh.

Đây chính là tinh thần greedy của ID3: mỗi bước chọn thuộc tính tốt nhất ngay lúc đó.

In [6]:
def id3(rows, attributes, target='play'):
    default = majority_label(rows, target=target)

    if all_same_label(rows, target=target):
        return TreeNode(label=rows[0][target], default_label=default)

    if not attributes:
        return TreeNode(label=default, default_label=default)

    best_attribute = max(attributes, key=lambda attr: information_gain(rows, attr, target=target))
    node = TreeNode(attribute=best_attribute, default_label=default)

    groups = partition_by_attribute(rows, best_attribute)
    remaining_attributes = [attr for attr in attributes if attr != best_attribute]

    for value, subset in groups.items():
        node.children[value] = id3(subset, remaining_attributes, target=target)

    return node


## Huấn luyện cây quyết định và in cấu trúc cây

Cell này sẽ dựng cây trên toàn bộ dữ liệu `weather` rồi in nó ra.

Nếu mọi thứ đúng, bạn sẽ thấy `outlook` thường được chọn ở node gốc vì nó có information gain cao nhất trên bộ dữ liệu này.

In [7]:
tree = id3(weather_data, attributes)
print(tree.pretty())

[outlook]
  (sunny)
    [humidity]
      (high)
        Predict -> no
      (normal)
        Predict -> yes
  (overcast)
    Predict -> yes
  (rainy)
    [wind]
      (weak)
        Predict -> yes
      (strong)
        Predict -> no


## Dự đoán cho vài mẫu mới

Sau khi có cây, mình thử dự đoán cho một vài tình huống mới.

Bạn có thể sửa các giá trị ở đây để tự kiểm tra cảm giác của mình về cách cây đang ra quyết định.

In [8]:
samples = [
    {'outlook': 'sunny', 'temperature': 'cool', 'humidity': 'high', 'wind': 'strong'},
    {'outlook': 'overcast', 'temperature': 'mild', 'humidity': 'high', 'wind': 'weak'},
    {'outlook': 'rainy', 'temperature': 'mild', 'humidity': 'normal', 'wind': 'weak'}
]

for sample in samples:
    print(sample, '->', tree.predict(sample))

{'outlook': 'sunny', 'temperature': 'cool', 'humidity': 'high', 'wind': 'strong'} -> no
{'outlook': 'overcast', 'temperature': 'mild', 'humidity': 'high', 'wind': 'weak'} -> yes
{'outlook': 'rainy', 'temperature': 'mild', 'humidity': 'normal', 'wind': 'weak'} -> yes


## Kiểm tra lại trên tập huấn luyện

Cell cuối cùng chỉ để kiểm tra nhanh xem cây vừa học có phân loại đúng lại dữ liệu training hay không.

Với bộ dữ liệu nhỏ và khá sạch như `weather`, cây thường sẽ học lại đúng toàn bộ tập huấn luyện.

In [9]:
y_true = [row['play'] for row in weather_data]
y_pred = [tree.predict(row) for row in weather_data]
accuracy = sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)
print('Training accuracy: %.2f %%' % (accuracy * 100))

Training accuracy: 100.00 %
